# OCR Pipeline 
This notebook extracts ingredients from skincare product label images using EasyOCR.

## Steps
1. Read image using EasyOCR
2. Extract and clean raw text
3. Split into individual ingredients
4. Return clean ingredient list

In [1]:
import easyocr
import re

reader = easyocr.Reader(['en'])

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [12]:
def extract_ingredients(image_path, confidence_threshold=0.3):
    
    # Step 1: OCR
    result = reader.readtext(image_path)
    lines = []
    for (bbox, text, confidence) in result:
        if confidence > confidence_threshold:
            lines.append(text)
    
    # Step 2: Join and basic cleaning
    raw_text = ' '.join(lines)

    ocr_corrections = {
        '1oo': '100',
        '10o': '100',
        '0o': '00',
        'o0': '00',
        '2O': '20',
        '- ': '-',
        '_': '',
    }

    for wrong, correct in ocr_corrections.items():
        raw_text = raw_text.replace(wrong, correct)

    raw_text = raw_text.replace(';', ',')
    raw_text = raw_text.strip()
    
    # Step 3: Find ingredients section
    raw_lower = raw_text.lower()
    if 'ingredient' in raw_lower:
        start = raw_lower.find('ingredient')
        start = raw_text.find(',', start)
        raw_text = raw_text[start+1:]
    
    # Step 4: Split by comma
    ingredients = raw_text.split(',')
    
    # Step 5: Clean each ingredient
    cleaned = []
    for ingredient in ingredients:
        ingredient = ingredient.strip()
        ingredient = ingredient.strip('.')
        ingredient = re.sub(r'^\d+\s*', '', ingredient)
        ingredient = re.sub(r'\s+', ' ', ingredient).strip()
        ingredient = ingredient.lower()
        if len(ingredient) > 2:
            cleaned.append(ingredient)
    
    # Step 6: Fix full stop merges
    further_cleaned = []
    for ingredient in cleaned:
        if '.' in ingredient:
            parts = ingredient.split('.')
            for part in parts:
                part = part.strip()
                if len(part) > 2:
                    further_cleaned.append(part)
        else:
            further_cleaned.append(ingredient)
    
    # Step 7: Fix ceramide type merges
    final_ingredients = []
    for ingredient in further_cleaned:
        if 'ceramide' in ingredient and ingredient.count('ceramide') > 1:
            parts = re.split(r'(?<!^)(?=ceramide)', ingredient)
            for part in parts:
                part = part.strip()
                if len(part) > 2:
                    final_ingredients.append(part)
        else:
            final_ingredients.append(ingredient)
    
    return final_ingredients

In [11]:
#Test 1 - Sunscreen Box
ingredients = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test1.jpg')
print(f"Total: {len(ingredients)}")
for i, ing in enumerate(ingredients, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Total: 32
1. cyclopentasiloxane
2. octyl salicylate
3. niacinamide
4. silica
5. avobenzone
6. octocrylene
7. butylene glycol
8. glycerine
9. homosalate
10. tapioca starch
11. neopentyl glycol diheptanoate
12. amps/hema crosspolymer (and) c13-15 alkane (and) coco-glucoside
13. octyldodecanol
14. polysorbate-20
15. zinc oxide
16. titanium dioxide
17. oats extract
18. zinc pca
19. cetearyl olivate & sorbitan olivate
20. ceramide ap
21. ceramide np
22. ceramide eos
23. d-panthenol
24. centella asiatica extract
25. ethylhexyl glycerine
26. phenoxyethanol
27. coco-glucoside
28. xanthan gum
29. tocopherol acetate
30. hyaluronic acid
31. citric acid
32. aqua


In [10]:
# Test 2 - Moisturizer Tube
ingredients_test2 = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test2.jpg')
print(f"Extracted: {len(ingredients_test2)}")
for i, ing in enumerate(ingredients_test2, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Extracted: 35
1. glycerin
2. sclerocarya birrea (marula) seed oil
3. methylpropanediol
4. dicaprylyl carbonate
5. caprylic/capric triglyceride
6. butylene glycol
7. cetearyl alcohol
8. glyceryl stearate
9. peg-1oo stearate
10. trehalose
11. dimethicone
12. saccharide isomerate
13. pentylene glycol
14. coco-caprylate/caprate
15. triheptanoin
16. c9-12 alkane
17. sodium hyaluronate
18. allantoin
19. linoleic acid
20. linolenic acid
21. carbomer
22. cetyl alcohol
23. phenoxyethanol
24. tocopheryl acetate
25. triethanolamine
26. nylon-12
27. petrolatum
28. hydroxyethyl acrylate/sodium acryloyldimethyl taurate copolymer
29. ethylhexylglycerin
30. butyrospermum parkii (shea) butter
31. polyurethane-10o
32. trisodium ethylenediamine disuccinate
33. tocopherol
34. citric acid
35. sodium citrate


In [14]:
# Test 3.1 - Face Mist Bottle
ingredients_test2 = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test3.1.jpg')
print(f"Extracted: {len(ingredients_test2)}")
for i, ing in enumerate(ingredients_test2, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Extracted: 6
1. ngredients: aqua
2. polysorbate 24 gycerin
3. almond oil
4. disodium edta camellia extract
5. witch hazel exdad white lotus extract
6. parfum


In [15]:
# Test 3.2 - Face Mist Bottle
ingredients_test2 = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test3.2.jpg')
print(f"Extracted: {len(ingredients_test2)}")
for i, ing in enumerate(ingredients_test2, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Extracted: 4
1. polysorbate 20
2. disodium edta camella extract
3. witch hazel extracl ilhite lotus extract
4. parfum


In [16]:
# Test 4 - Face Wash Tube
ingredients_test2 = extract_ingredients('C:/Users/anamt/OneDrive/Desktop/skincare-analyzer/test_images/test4.jpg')
print(f"Extracted: {len(ingredients_test2)}")
for i, ing in enumerate(ingredients_test2, 1):
    print(f"{i}. {ing}")

C:\Users\anamt\OneDrive\Desktop\skincare-analyzer\venv310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Extracted: 19
1. igredients: aqua
2. cocamidopropyl betaine
3. acrylates copolyme} coco glucoside
4. sodium cocoyl glycinate
5. glycerin
6. niacinamide
7. jrethanolamine
8. rice ferment filtrate (sake)
9. oryza sativa (rice) (etract
10. capryloylcaproyl methyl glucamide
11. lauroylmyristoyl ethyl glucamide
12. phenoxyethanol
13. ethylhexylglycerin
14. olive oil styrene/acrylates copolymer
15. disodium edta
16. fragrance (ifra-certified)
17. panthenol
18. butylene gla acetate sodiue peg-7 esters
19. aaluronate
